# Build a Personal Email-Triage Agent — Colab / Kaggle / Binder notebook

This notebook is a zero-install way to run the [Build a Personal Email-Triage Agent](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/email-triage-agent) project — no local Python install, no cloned repo, no `.env` file. It mirrors the lesson's steps 1-4 exactly, using the same `sample_emails/` content and the same `triage.py` logic from [`examples/email-triage-agent/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/email-triage-agent) in the course repo.

**Safety boundary (same as the lesson):** this notebook never sends email. There is no SMTP code anywhere here — not commented out, not behind a flag. Drafts are only printed, never sent.

**For the primary "graduate to real Python" experience**, do this project locally with `uv` instead — see the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/email-triage-agent) for that path. This notebook is a lower-fidelity but genuinely complete way to run the same pipeline with zero setup, handy for a quick look on Colab, Kaggle, or Binder.


## Step 0: Install dependencies

The project only needs the `openai` package (used as a generic client for any OpenAI-compatible Chat Completions endpoint — every free-tier provider in the lesson exposes one). We skip `python-dotenv` here since a notebook has no local `.env` file to read; instead we'll collect the API key interactively in the next step.

In [ ]:
!pip install -q openai>=1.59.0


## Step 1: Pick a provider and enter your free-tier API key

Just like the lesson, pick **one** free-tier provider and get a key from its site (see the lesson's ["Get a free AI API key"](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/email-triage-agent#get-a-free-ai-api-key) section for links and details: GitHub Models, Gemini, Groq, Mistral, Cerebras, or OpenRouter).

Because this is a notebook, there's no local `.env` file to read the key from — instead, `getpass.getpass()` prompts for it interactively below, and it's kept only in this runtime's memory (`os.environ`), never written to disk or shown on screen as you type. **Never paste an API key directly into a code cell as a string literal** — always enter it through the prompt below.

In [ ]:
import os
from getpass import getpass

# Pick one: "github" (suggested default), "gemini", "groq", "mistral", "cerebras", or "openrouter"
LLM_PROVIDER = "github"  # change this if you picked a different provider
os.environ["LLM_PROVIDER"] = LLM_PROVIDER

PROVIDERS = {
    "github": {
        "base_url": "https://models.github.ai/inference",
        "api_key_env": "GITHUB_TOKEN",
        "model": "gpt-4o-mini",
    },
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key_env": "GOOGLE_API_KEY",
        "model": "gemini-3.5-flash",
    },
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "api_key_env": "GROQ_API_KEY",
        "model": "llama-3.3-70b-versatile",
    },
    "mistral": {
        "base_url": "https://api.mistral.ai/v1",
        "api_key_env": "MISTRAL_API_KEY",
        "model": "mistral-small-latest",
    },
    "cerebras": {
        "base_url": "https://api.cerebras.ai/v1",
        "api_key_env": "CEREBRAS_API_KEY",
        "model": "llama-3.3-70b",
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "model": "meta-llama/llama-3.3-70b-instruct:free",
    },
}

if LLM_PROVIDER not in PROVIDERS:
    raise ValueError(f"Unknown LLM_PROVIDER '{LLM_PROVIDER}'. Choose one of: {', '.join(PROVIDERS)}")

_config = PROVIDERS[LLM_PROVIDER]
_key_env = _config["api_key_env"]

if not os.environ.get(_key_env):
    os.environ[_key_env] = getpass(f"Paste your {LLM_PROVIDER} API key ({_key_env}): ")

print(f"Provider set to '{LLM_PROVIDER}', using model '{_config['model']}'.")


## Step 2: Build the OpenAI-compatible client

Every provider above exposes an OpenAI-compatible Chat Completions endpoint, so one small client class covers all six — only the `base_url`, model name, and which environment variable holds the key change. This is the same `build_client` function from the repo's `triage.py`.

In [ ]:
from openai import OpenAI


def build_client() -> tuple[OpenAI, str]:
    """Builds an OpenAI-compatible client for LLM_PROVIDER (env var, default
    "github"). Returns (client, model_name)."""
    provider = os.environ.get("LLM_PROVIDER", "github")
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    config = PROVIDERS[provider]
    api_key = os.environ.get(config["api_key_env"])
    if not api_key:
        raise KeyError(
            f"{config['api_key_env']} is not set. Re-run the previous cell and paste a key."
        )
    client = OpenAI(api_key=api_key, base_url=config["base_url"])
    return client, config["model"]


client, model = build_client()
print(f"Client ready for model '{model}'.")


## Step 1 (lesson): The sample emails

The repo example ships six short, realistic sample emails in `sample_emails/` — an urgent client request, a newsletter, two messages that genuinely need a reply, a spammy promo, and an automated FYI notification. Since a notebook running on Colab/Kaggle/Binder has no guaranteed access to the course repo's files, they're embedded directly below as plain strings — the exact same content as the six `.txt` files in [`examples/email-triage-agent/sample_emails/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/email-triage-agent/sample_emails), shaped like a simplified `.eml` file: a few `Header: value` lines, a blank line, then the body.

In [ ]:
SAMPLE_EMAILS = {
    "01_urgent_client.txt": """From: maria.chen@brightpath-consulting.com
To: you@example.com
Subject: URGENT: Contract needs your signature before 5pm today
Date: Mon, 20 Jul 2026 09:14:00 -0400

Hi,

Sorry for the short notice, but legal just flagged that the vendor
agreement needs to be signed and returned before end of day today or we
lose the discounted rate we negotiated. I've attached the final version
(no changes from what we discussed on Thursday's call).

Can you sign and send it back before 5pm? If you're not able to, please
let me know as soon as possible so I can ask them for a short extension.

Thanks so much,
Maria
""",
    "02_newsletter.txt": """From: newsletter@pythonweekly.example.com
To: you@example.com
Subject: Python Weekly #612: pattern matching, faster pandas, and more
Date: Sun, 19 Jul 2026 07:00:00 -0400

This week in Python:

- A deep dive on structural pattern matching (match/case) beyond the basics
- Benchmarks: the new pandas release is up to 30% faster on groupby
- Five underrated standard library modules you're probably not using
- Job board: 40 new remote Python roles posted this week
- Upcoming conferences and meetups near you

Read the full issue on our website. You're receiving this because you
subscribed at pythonweekly.example.com. Unsubscribe anytime from the link
in the footer.
""",
    "03_needs_reply_coworker.txt": """From: sam.okoye@yourcompany.example.com
To: you@example.com
Subject: Quick question about the Q3 numbers before I present tomorrow
Date: Mon, 20 Jul 2026 14:32:00 -0400

Hey,

I'm putting together the slides for tomorrow's Q3 review and the revenue
figure in the shared spreadsheet doesn't match what you sent me last
week -- the spreadsheet says 482K but your email said 512K. Which one is
right? I want to make sure I don't put a wrong number in front of the
leadership team.

Also, quick unrelated thing: are you around for a 15-minute sync
tomorrow morning before the review, just to make sure we're aligned on
talking points?

Thanks,
Sam
""",
    "04_spammy_promo.txt": """From: deals@totally-legit-savings.example.net
To: you@example.com
Subject: You've been SELECTED for a FREE $500 gift card!!! Click now
Date: Mon, 20 Jul 2026 03:47:00 -0400

CONGRATULATIONS!!!

Our system has randomly selected your email for a FREE $500 gift card
from a major retailer. This offer expires in 24 HOURS so act now!

Click the link below and enter your details to claim your prize:
[CLAIM YOUR REWARD NOW]

This is a limited time offer and will not be repeated. Don't miss out!

If you no longer wish to receive these amazing offers, well, that's a
shame, but there's a link somewhere below, good luck finding it.
""",
    "05_needs_reply_scheduling.txt": """From: priya.nair@partnerorg.example.com
To: you@example.com
Subject: Rescheduling our Thursday call
Date: Tue, 21 Jul 2026 10:05:00 -0400

Hi there,

Something came up on my end and I won't be able to make our call this
Thursday at 2pm. Would any of these alternative times work for you
instead?

- Friday at 10am
- Monday at 1pm
- Monday at 3:30pm

Let me know which one fits your schedule, or feel free to suggest a
different time if none of these work. Sorry for the shuffle!

Best,
Priya
""",
    "06_fyi_no_reply.txt": """From: notifications@teamboard.example.com
To: you@example.com
Subject: [TeamBoard] Your weekly project summary is ready
Date: Sun, 19 Jul 2026 23:00:00 -0400

Hi,

Here's your automated weekly summary for the "Website Redesign" board:

- 8 tasks completed this week
- 3 tasks moved to "In Review"
- 1 task overdue: "Finalize color palette" (assigned to Jordan)

No action is needed from you -- this is an automated summary email sent
every Sunday. You can adjust your notification preferences in your
account settings.
""",
}

print(f"Embedded {len(SAMPLE_EMAILS)} sample emails.")


Now parse them with the same `Email` dataclass and `parse_email` logic as the lesson's `triage.py` — just reading from the in-memory `SAMPLE_EMAILS` dict instead of the filesystem.

In [ ]:
from dataclasses import dataclass


@dataclass
class Email:
    filename: str
    sender: str
    subject: str
    date: str
    body: str


def parse_email(filename: str, text: str) -> Email:
    """Parses one plain-text sample email: a few `Header: value` lines, a
    blank line, then the body -- the same shape as a real .eml file's
    headers, simplified so no email-parsing library is needed for the
    bundled samples."""
    header_text, _, body = text.partition("\n\n")
    headers = {}
    for line in header_text.splitlines():
        if ":" in line:
            key, _, value = line.partition(":")
            headers[key.strip().lower()] = value.strip()
    return Email(
        filename=filename,
        sender=headers.get("from", "unknown"),
        subject=headers.get("subject", "(no subject)"),
        date=headers.get("date", "unknown"),
        body=body.strip(),
    )


emails = [parse_email(name, text) for name, text in sorted(SAMPLE_EMAILS.items())]

print(f"Loaded {len(emails)} email(s)\n")
for email in emails:
    print(f"[{email.filename}] {email.subject!r} from {email.sender}")


## Step 2 (lesson): Categorize and prioritize each email with an LLM

Hand each parsed email to the model and ask it to sort it into a category and a priority — the same `TRIAGE_PROMPT` and `triage_email` function as `triage.py`.

In [ ]:
import json

TRIAGE_PROMPT = """You are an email triage assistant. Read the email below and respond with ONLY a JSON object (no other text, no markdown fence), with these exact keys:

- "category": one of "urgent", "needs-reply", "newsletter", "fyi", "spam-ish"
- "priority": one of "high", "medium", "low"
- "reasoning": one short sentence explaining the category and priority
- "needs_reply": true or false

Email:
From: {sender}
Subject: {subject}
Date: {date}

{body}
"""


def triage_email(client: OpenAI, model: str, email: Email) -> dict:
    """Asks the LLM to categorize and prioritize one email. Returns the
    parsed JSON verdict. Read-only: never modifies or sends anything."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": TRIAGE_PROMPT.format(
                    sender=email.sender, subject=email.subject, date=email.date, body=email.body
                ),
            }
        ],
    )
    content = response.choices[0].message.content.strip()
    # Models occasionally wrap JSON in a ```json fence despite the prompt --
    # strip it rather than fail outright.
    if content.startswith("```"):
        content = content.strip("`")
        content = content.removeprefix("json").strip()
    return json.loads(content)


## Step 3 (lesson): Draft (but never send) a reply

For anything the model marked `needs_reply: true`, ask it to draft an actual reply. **This function only ever returns a string.** There is no SMTP code anywhere in this notebook — not commented out, not behind a flag — the same hard boundary as the lesson and the repo's `triage.py`.

In [ ]:
DRAFT_REPLY_PROMPT = """Draft a short, professional reply to the email below. Write ONLY the reply body text -- no subject line, no commentary about what you're doing, just the reply itself, as if the recipient is about to review and send it.

Original email:
From: {sender}
Subject: {subject}

{body}
"""


def draft_reply(client: OpenAI, model: str, email: Email) -> str:
    """Asks the LLM to draft a reply. The result is ALWAYS just printed
    here -- this function has no way to actually send anything, on purpose."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": DRAFT_REPLY_PROMPT.format(
                    sender=email.sender, subject=email.subject, body=email.body
                ),
            }
        ],
    )
    return response.choices[0].message.content.strip()


## Step 4 (lesson): Run it end to end

Wire everything together: categorize every email, and draft a reply for the ones that need one. On Colab/Kaggle there's no guarantee the notebook's working directory persists between sessions, so instead of writing files to a local `drafts/` folder like `triage.py` does, this cell just prints every draft directly — read it the same way you'd read a saved file, and remember: nothing here is ever sent anywhere.

In [ ]:
print(f"Triaging {len(emails)} email(s) with model '{model}'...\n")

drafts = {}

for email in emails:
    verdict = triage_email(client, model, email)
    print(f"[{email.filename}] {email.subject!r}")
    print(f"  category: {verdict['category']}   priority: {verdict['priority']}")
    print(f"  reasoning: {verdict['reasoning']}")

    if verdict.get("needs_reply"):
        reply = draft_reply(client, model, email)
        drafts[email.filename] = reply
        print("  -> draft reply generated (NOT sent -- review and send yourself)")
    print()

print(
    "Done. This notebook only ever printed text -- it has no code path that "
    "sends email anywhere. Read every draft yourself before copying any of "
    "it into a real reply.\n"
)

for filename, reply in drafts.items():
    print(f"===== Draft reply for {filename} =====")
    print(reply)
    print()


## Where to go from here

- Run this on your own emails: see the lesson's "Optional, go further" section — but do that part **locally**, never in a notebook, since it involves a real (app) password.
- For the full "graduate to real Python" experience with a real local project, `uv`, and a proper `drafts/` folder, follow the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/email-triage-agent) from the top with `uv` instead of this notebook.
- Read [`examples/email-triage-agent/README.md`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/email-triage-agent) in the course repo for the local-run instructions and safety notes this notebook mirrors.